In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

print("PyTorch version:", torch.__version__)

PyTorch version: 2.5.1


### 1) Core primitives: add/remove dims, reshape, transpose, contiguous

In [2]:
# Start with a 3D tensor representing (Batch, Channels, Length)
x = torch.randn(2, 3, 4)  # Shape: (B=2, C=3, L=4)
print(f"Original tensor shape: {x.shape}")

Original tensor shape: torch.Size([2, 3, 4])


In [3]:
# ADD DIMENSIONS with unsqueeze()
# unsqueeze(0) adds a new dimension at position 0 (beginning)
y = x.unsqueeze(0)  # Insert new dim at index 0: (2,3,4) -> (1,2,3,4)
print(f"After unsqueeze(0): {y.shape}")  # (1, 2, 3, 4)
# This is often used to add batch dimension to single samples

After unsqueeze(0): torch.Size([1, 2, 3, 4])


In [4]:
# unsqueeze(-1) adds dimension at the end (last position)
z = x.unsqueeze(-1)  # Insert new dim at end: (2,3,4) -> (2,3,4,1)
print(f"After unsqueeze(-1): {z.shape}")  # (2, 3, 4, 1)
# Common for adding feature dimension or preparing for broadcasting

After unsqueeze(-1): torch.Size([2, 3, 4, 1])


In [5]:
# REMOVE DIMENSIONS with squeeze()
# squeeze() removes all dimensions of size 1
w = z.squeeze(-1)  # Remove last dim if size=1: (2,3,4,1) -> (2,3,4)
print(f"After squeeze(-1): {w.shape}")  # (2, 3, 4)

After squeeze(-1): torch.Size([2, 3, 4])


In [6]:
# RESHAPE operations - changing tensor shape while preserving total elements
# Total elements must remain the same: 2*3*4 = 24 elements
a = x.reshape(6, 4)  # Reshape to (6,4): groups first two dims together
print(f"Reshape to (6,4): {a.shape}")
# reshape() is SAFE - it will copy data if needed to maintain contiguity

Reshape to (6,4): torch.Size([6, 4])


In [7]:
# view() is FASTER but requires contiguous memory layout
b = x.view(6, 4)  # Same result as reshape but requires contiguous tensor
print(f"View to (6,4): {b.shape}")

View to (6,4): torch.Size([6, 4])


In [8]:
# PERMUTE changes dimension order (like transpose but for N-D tensors)
c = x.permute(1, 0, 2)  # Reorder dims: (B,C,L) -> (C,B,L)
print(f"After permute(1,0,2): {c.shape}")  # (3, 2, 4)
print(f"Is permuted tensor contiguous? {c.is_contiguous()}")  # Usually False

After permute(1,0,2): torch.Size([3, 2, 4])
Is permuted tensor contiguous? False


In [9]:
# After permute, tensor is often NON-CONTIGUOUS in memory
# Need .contiguous() before using .view()
try:
    c.view(3, 8)  # This might fail because c is not contiguous
    print("View after permute: SUCCESS")
except:
    print("View after permute: FAILED (not contiguous)")

View after permute: FAILED (not contiguous)


In [10]:
# Solution: make it contiguous first
c_contig = c.contiguous().view(3, 8)  # Now it works
print(f"Contiguous then view: {c_contig.shape}")

Contiguous then view: torch.Size([3, 8])


In [11]:
# FLATTEN - convenient way to collapse dimensions
flat = x.flatten(start_dim=1)  # Keep dim 0 (batch), flatten rest: (2,3,4) -> (2,12)
print(f"Flatten from dim 1: {flat.shape}")  # (2, 12)

Flatten from dim 1: torch.Size([2, 12])


In [12]:
# Basic example: flatten to 1D
x = torch.randn(2, 3, 4)  # Total elements: 2*3*4 = 24
print(f"Original tensor: {x.shape}")

# reshape(-1) flattens to 1D - most common usage
flattened = x.reshape(-1)
print(f"reshape(-1): {x.shape} -> {flattened.shape}")
print(f"Total elements preserved: {x.numel()} = {flattened.numel()}")
print()

Original tensor: torch.Size([2, 3, 4])
reshape(-1): torch.Size([2, 3, 4]) -> torch.Size([24])
Total elements preserved: 24 = 24



In [13]:
# =============================================================================
# reshape(batch, -1): Keep batch dimension, flatten rest
# =============================================================================

batch_data = torch.randn(8, 3, 32, 32)  # Batch of images: (B, C, H, W)
print(f"Batch of images: {batch_data.shape}")

# Keep batch dim, flatten spatial+channel dims: (8, 3*32*32)
batch_flattened = batch_data.reshape(8, -1)
print(f"reshape(8, -1): {batch_data.shape} -> {batch_flattened.shape}")
print(f"Each sample now has {batch_flattened.shape[1]} features")
print()

# Alternative: use batch_size variable for flexibility
batch_size = batch_data.shape[0]
flexible_flatten = batch_data.reshape(batch_size, -1)
print(f"reshape(batch_size, -1): {flexible_flatten.shape}")
print()

Batch of images: torch.Size([8, 3, 32, 32])
reshape(8, -1): torch.Size([8, 3, 32, 32]) -> torch.Size([8, 3072])
Each sample now has 3072 features

reshape(batch_size, -1): torch.Size([8, 3072])



In [14]:
# =============================================================================
# reshape(-1, feature_dim): Infer first dimension
# =============================================================================

embeddings = torch.randn(4, 10, 128)  # (batch, seq_len, embed_dim)
print(f"Embeddings tensor: {embeddings.shape}")

# Combine batch and sequence dimensions: (4*10, 128)
combined = embeddings.reshape(-1, 128)
print(f"reshape(-1, 128): {embeddings.shape} -> {combined.shape}")
print(f"Now have {combined.shape[0]} individual embeddings of size {combined.shape[1]}")
print()

Embeddings tensor: torch.Size([4, 10, 128])
reshape(-1, 128): torch.Size([4, 10, 128]) -> torch.Size([40, 128])
Now have 40 individual embeddings of size 128



In [15]:
# 1. Preparing image data for linear layers
image_batch = torch.randn(32, 3, 224, 224)  # ImageNet-style batch
print(f"1. Image batch: {image_batch.shape}")
for_linear = image_batch.reshape(32, -1)     # Flatten for FC layer
print(f"   For linear layer: {for_linear.shape}")
print(f"   Each image -> {for_linear.shape[1]} pixel values")
print()

1. Image batch: torch.Size([32, 3, 224, 224])
   For linear layer: torch.Size([32, 150528])
   Each image -> 150528 pixel values



In [16]:
# 2. Reshaping attention outputs
seq_len, embed_dim, num_heads = 16, 512, 8
head_dim = embed_dim // num_heads  # 64
attn_output = torch.randn(2, num_heads, seq_len, head_dim)  # Multi-head output
print(f"2. Multi-head attention: {attn_output.shape}")
combined_heads = attn_output.reshape(2, seq_len, -1)  # Combine heads
print(f"   Combined heads: {combined_heads.shape}")
print(f"   Reconstructed embed_dim: {combined_heads.shape[2]}")
print()

2. Multi-head attention: torch.Size([2, 8, 16, 64])
   Combined heads: torch.Size([2, 16, 512])
   Reconstructed embed_dim: 512



In [17]:
# 3. Preparing sequences for RNN processing
batch_texts = torch.randn(5, 20, 300)  # (batch, max_seq_len, word_vec_dim)
print(f"3. Batch of text sequences: {batch_texts.shape}")
all_words = batch_texts.reshape(-1, 300)  # All words in single tensor
print(f"   All words flattened: {all_words.shape}")
print(f"   Total words across all sequences: {all_words.shape[0]}")
print()

3. Batch of text sequences: torch.Size([5, 20, 300])
   All words flattened: torch.Size([100, 300])
   Total words across all sequences: 100



In [18]:
print("=== reshape(-1) vs alternatives ===")

data = torch.randn(2, 3, 4)
print(f"Original: {data.shape}")

# Method 1: reshape(-1) - SAFE, handles non-contiguous tensors
method1 = data.reshape(-1)
print(f"reshape(-1): {method1.shape}")

# Method 2: flatten() - equivalent to reshape(-1) for full flattening
method2 = data.flatten()
print(f"flatten(): {method2.shape}")

# Method 3: view(-1) - FAST but requires contiguous memory
method3 = data.view(-1)
print(f"view(-1): {method3.shape}")

# Test with non-contiguous tensor
permuted = data.permute(2, 0, 1)  # Makes tensor non-contiguous
print(f"\nAfter permute: {permuted.shape}, contiguous: {permuted.is_contiguous()}")

# reshape(-1) works even with non-contiguous tensors
safe_reshape = permuted.reshape(-1)
print(f"reshape(-1) on non-contiguous: ✓ Works, shape: {safe_reshape.shape}")

# view(-1) might fail on non-contiguous tensors
try:
    risky_view = permuted.view(-1)
    print(f"view(-1) on non-contiguous: ✓ Works, shape: {risky_view.shape}")
except:
    print("view(-1) on non-contiguous: ✗ Failed - need .contiguous() first")
    fixed_view = permuted.contiguous().view(-1)
    print(f"permuted.contiguous().view(-1): ✓ Works, shape: {fixed_view.shape}")

print()

=== reshape(-1) vs alternatives ===
Original: torch.Size([2, 3, 4])
reshape(-1): torch.Size([24])
flatten(): torch.Size([24])
view(-1): torch.Size([24])

After permute: torch.Size([4, 2, 3]), contiguous: False
reshape(-1) on non-contiguous: ✓ Works, shape: torch.Size([24])
view(-1) on non-contiguous: ✗ Failed - need .contiguous() first
permuted.contiguous().view(-1): ✓ Works, shape: torch.Size([24])



### 2) Permute/transpose vs reshape - when to use which


In [19]:
# PERMUTE: reorders dimensions without changing element relationships
# Common use: converting between different tensor formats
x_img = torch.randn(8, 32, 224, 224)  # (Batch, Channels, Height, Width)
print(f"Image tensor (NCHW): {x_img.shape}")

Image tensor (NCHW): torch.Size([8, 32, 224, 224])


In [20]:
# Convert to channels-last format (NHWC) - common for some operations
x_channels_last = x_img.permute(0, 2, 3, 1)  # (B,C,H,W) -> (B,H,W,C)
print(f"Channels last (NHWC): {x_channels_last.shape}")
# Must call contiguous() if you plan to use view() later
x_channels_last = x_channels_last.contiguous()

Channels last (NHWC): torch.Size([8, 224, 224, 32])


### 3) Indexing: slices, fancy indexing, boolean masks

In [21]:
# Create example tensor with known values for easier debugging
x = torch.arange(24).view(2, 3, 4)  # Shape (2,3,4) with values 0-23
print(f"Example tensor shape: {x.shape}")
print(f"Example tensor:\n{x}")

Example tensor shape: torch.Size([2, 3, 4])
Example tensor:
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])


In [23]:
# SLICE INDEXING - creates views (no copying)
slice_result = x[:, 1, :]  # All batches, channel 1, all positions -> (2,4)
print(f"Slice [:, 1, :] shape: {slice_result.shape}")
print(f"Slice result:\n{slice_result}")

Slice [:, 1, :] shape: torch.Size([2, 4])
Slice result:
tensor([[ 4,  5,  6,  7],
        [16, 17, 18, 19]])


In [24]:
# FANCY INDEXING - uses tensors as indices (creates copies, not views)
rows = torch.tensor([0, 1])  # Select these batch indices
cols = torch.tensor([2, 0])  # Select these position indices
# Advanced indexing semantics: broadcasting applies
fancy_result = x[rows, :, cols]  # Shape depends on broadcasting rules
print(f"Fancy indexing [rows, :, cols] shape: {fancy_result.shape}")
print(f"Fancy indexing result:\n{fancy_result}")

Fancy indexing [rows, :, cols] shape: torch.Size([2, 3])
Fancy indexing result:
tensor([[ 2,  6, 10],
        [12, 16, 20]])


In [25]:
# BOOLEAN MASKING - select elements based on condition
mask = x > 10  # Boolean tensor same shape as x
print(f"Mask shape: {mask.shape}")
selected = x[mask]  # Returns 1D tensor of selected elements
print(f"Boolean mask result shape: {selected.shape}")
print(f"Selected values (>10): {selected}")

Mask shape: torch.Size([2, 3, 4])
Boolean mask result shape: torch.Size([13])
Selected values (>10): tensor([11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23])


In [35]:
# CONDITIONAL SELECTION with torch.where
# where(condition, value_if_true, value_if_false)
conditional = torch.where(x > 10, x, torch.zeros_like(x))  # Keep shape
print(f"torch.where result shape: {conditional.shape}")
print(f"torch.where result (replace <=10 with 0):\n{conditional}")

torch.where result shape: torch.Size([2, 3, 4])
torch.where result (replace <=10 with 0):
tensor([[[ 0,  0,  0,  0],
         [ 0,  0,  0,  0],
         [ 0,  0,  0, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])


### 4) Gather/scatter - batched selection & writingm

In [27]:
# GATHER - batched indexing for reading values
# Common use case: embedding lookup with different indices per batch
B, N, D = 2, 10, 16  # Batch size, vocab size, embedding dim
K = 4  # Number of tokens to select per batch

In [28]:
# Create embedding table: each batch has its own embeddings
emb = torch.randn(B, N, D)  # (Batch, Vocab_size, Embedding_dim)
print(f"Embedding table shape: {emb.shape}")

Embedding table shape: torch.Size([2, 10, 16])


In [34]:
# Indices to gather: different tokens per batch
indices = torch.randint(0, N, (B, K))  # (Batch, Sequence_length), here low = 0, high = 10, and shape will be (B, K) = 2, 4
print(indices.shape)
print(f"Indices to gather: {indices}")

torch.Size([2, 4])
Indices to gather: tensor([[0, 6, 5, 9],
        [6, 0, 6, 2]])


In [32]:
indices.unsqueeze(-1).shape

torch.Size([2, 4, 1])

In [35]:
# IMPORTANT: gather requires index tensor to have same number of dims as source
# We need to expand indices to match embedding dimension
idx_expanded = indices.unsqueeze(-1).expand(-1, -1, D)  # (B, K, D) -1 in the first two positions → keep existing sizes (B, K).
print(f"Expanded indices shape: {idx_expanded.shape}")

Expanded indices shape: torch.Size([2, 4, 16])


In [36]:
# Gather along vocabulary dimension (dim=1)
gathered = torch.gather(emb, dim=1, index=idx_expanded)  # (B, K, D)
print(f"Gathered embeddings shape: {gathered.shape}")

Gathered embeddings shape: torch.Size([2, 4, 16])


In [37]:
# SCATTER - batched writing/accumulation
# Common use case: accumulating gradients or features to specific positions
B, K, D, N = 2, 5, 8, 10  # Batch, sequence, features, target_positions
src = torch.randn(B, K, D)  # Source features to scatter: (B, K, D)
idx = torch.randint(0, N, (B, K))  # Where to place each feature: (B, K)
print(f"Source features shape: {src.shape}")
print(f"Scatter indices: {idx}")

Source features shape: torch.Size([2, 5, 8])
Scatter indices: tensor([[3, 3, 3, 4, 8],
        [1, 4, 8, 6, 4]])


In [38]:
# Initialize target tensor (often zeros for accumulation)
target = torch.zeros(B, N, D)
print(f"Target tensor shape: {target.shape}")

Target tensor shape: torch.Size([2, 10, 8])


In [40]:
# Expand indices to match feature dimension
idx_expanded = idx.unsqueeze(-1).expand(-1, -1, D)  # (B, K, D)
idx_expanded.shape

torch.Size([2, 5, 8])

In [41]:
# Scatter-add: accumulate source features into target positions
# scatter_add_(dim, index, source) modifies tensor in-place
target.scatter_add_(1, idx_expanded, src)  # Accumulate along dim=1
print(f"After scatter_add, target shape: {target.shape}")

After scatter_add, target shape: torch.Size([2, 10, 8])


### 5) Batched matmul, bmm, and einsum

In [42]:
# BATCHED MATRIX MULTIPLICATION with broadcasting
A = torch.randn(2, 3, 4)  # (Batch=2, Rows=3, Inner=4)
B = torch.randn(4, 5)     # (Inner=4, Cols=5) - will broadcast to (2,4,5)
print(f"A shape: {A.shape}, B shape: {B.shape}")

A shape: torch.Size([2, 3, 4]), B shape: torch.Size([4, 5])


In [43]:
# torch.matmul handles broadcasting automatically
result = torch.matmul(A, B)  # (2,3,4) @ (4,5) -> (2,3,5)
print(f"Matmul result shape: {result.shape}")

Matmul result shape: torch.Size([2, 3, 5])


In [44]:
# BMM - strict batched matrix multiplication (no broadcasting)
X = torch.randn(8, 10, 16)  # (Batch, Rows, Inner)
Y = torch.randn(8, 16, 12)  # (Batch, Inner, Cols)
Z = torch.bmm(X, Y)         # (Batch, Rows, Cols)
print(f"BMM: {X.shape} @ {Y.shape} = {Z.shape}")

BMM: torch.Size([8, 10, 16]) @ torch.Size([8, 16, 12]) = torch.Size([8, 10, 12])


In [45]:
# EINSUM - explicit notation for complex tensor operations
# Einstein summation: repeated indices are summed over
# 'bij,bjk->bik' means: for each batch b, multiply i,j with j,k -> i,k
einsum_result = torch.einsum('bij,bjk->bik', X, Y)
print(f"Einsum result shape: {einsum_result.shape}")

Einsum result shape: torch.Size([8, 10, 12])


In [46]:
# ATTENTION SCORE COMPUTATION with einsum
# Q: queries (B, Heads, Query_len, Key_dim)
# K: keys (B, Heads, Key_len, Key_dim)
B, H, Q_len, K_len, dk = 2, 8, 16, 20, 64
Q = torch.randn(B, H, Q_len, dk)  # Queries
K = torch.randn(B, H, K_len, dk)  # Keys
print(f"Q shape: {Q.shape}, K shape: {K.shape}")

Q shape: torch.Size([2, 8, 16, 64]), K shape: torch.Size([2, 8, 20, 64])


In [47]:
# Compute attention scores: each query attends to each key
# 'bhqd,bhkd->bhqk': batch, head, (query_pos, key_dim) * (key_pos, key_dim) -> (query_pos, key_pos)
attention_scores = torch.einsum('bhqd,bhkd->bhqk', Q, K)
print(f"Attention scores shape: {attention_scores.shape}")  # (B, H, Q_len, K_len)

Attention scores shape: torch.Size([2, 8, 16, 20])


### 6) Multi-head Attention - complete implementation

In [48]:
# Setup dimensions
B, S, D, H = 2, 16, 64, 8  # Batch, Sequence, Model_dim, Heads
dk = D // H  # Key dimension per head: 64/8 = 8
print(f"Batch: {B}, Sequence: {S}, Model_dim: {D}, Heads: {H}, Key_dim_per_head: {dk}")

Batch: 2, Sequence: 16, Model_dim: 64, Heads: 8, Key_dim_per_head: 8


In [49]:
# Input sequence
x = torch.randn(B, S, D)  # (Batch, Sequence, Features)
print(f"Input shape: {x.shape}")

Input shape: torch.Size([2, 16, 64])


In [50]:
# Linear projection weights (normally these would be nn.Linear layers)
Wq = torch.randn(D, D)  # Query projection
Wk = torch.randn(D, D)  # Key projection
Wv = torch.randn(D, D)  # Value projection

# Project to queries, keys, values
q = x @ Wq  # (B, S, D) @ (D, D) -> (B, S, D)
k = x @ Wk  # (B, S, D)
v = x @ Wv  # (B, S, D)

In [51]:
print(f"After projection - Q: {q.shape}, K: {k.shape}, V: {v.shape}")

After projection - Q: torch.Size([2, 16, 64]), K: torch.Size([2, 16, 64]), V: torch.Size([2, 16, 64])


In [52]:
# SPLIT INTO MULTIPLE HEADS
def split_heads(tensor):
    """Reshape (B, S, D) -> (B, H, S, dk) for multi-head attention"""
    # First reshape to separate heads: (B, S, D) -> (B, S, H, dk)
    reshaped = tensor.view(B, S, H, dk)
    # Then move heads to second dimension: (B, S, H, dk) -> (B, H, S, dk)
    return reshaped.permute(0, 2, 1, 3)

Q = split_heads(q)  # (B, H, S, dk)
K = split_heads(k)  # (B, H, S, dk)
V = split_heads(v)  # (B, H, S, dk)
print(f"After head splitting - Q: {Q.shape}, K: {K.shape}, V: {V.shape}")


After head splitting - Q: torch.Size([2, 8, 16, 8]), K: torch.Size([2, 8, 16, 8]), V: torch.Size([2, 8, 16, 8])


In [54]:
# COMPUTE ATTENTION SCORES
# Each query position attends to each key position
# Transpose last 2 dims of K: (B, H, S, dk) -> (B, H, dk, S)
scores = torch.matmul(Q, K.transpose(-2, -1))  # (B, H, S, S)
# Scale by sqrt(dk) to prevent softmax saturation
scores = scores / math.sqrt(dk)
print(f"Attention scores shape: {scores.shape}")

Attention scores shape: torch.Size([2, 8, 16, 16])


In [55]:
# APPLY CAUSAL MASK (for autoregressive models)
# Create mask to prevent attending to future positions
mask = torch.ones(B, S).bool()  # (B, S) - True means "keep"
print(f"Initial mask shape: {mask.shape}")

Initial mask shape: torch.Size([2, 16])


In [56]:
# Expand mask to match scores dimensions
# (B, S) -> (B, 1, 1, S) to broadcast to (B, H, S, S)
mask = mask.unsqueeze(1).unsqueeze(2)  # Add head and query dimensions
print(f"Expanded mask shape: {mask.shape}")

Expanded mask shape: torch.Size([2, 1, 1, 16])


In [57]:
# Apply mask: set masked positions to -inf (will become 0 after softmax)
scores = scores.masked_fill(~mask, float("-inf"))
print(f"Scores after masking: min={scores.min():.2f}, max={scores.max():.2f}")

Scores after masking: min=-332.71, max=290.88


In [58]:
# SOFTMAX to get attention weights
A = torch.softmax(scores, dim=-1)  # (B, H, S, S)
print(f"Attention weights shape: {A.shape}")
print(f"Attention weights sum along last dim: {A.sum(dim=-1)[0, 0, 0]:.4f}")  # Should be ~1.0

Attention weights shape: torch.Size([2, 8, 16, 16])
Attention weights sum along last dim: 1.0000


In [59]:
# APPLY ATTENTION to values
out = torch.matmul(A, V)  # (B, H, S, S) @ (B, H, S, dk) -> (B, H, S, dk)
print(f"Attention output shape: {out.shape}")

Attention output shape: torch.Size([2, 8, 16, 8])


In [60]:
# COMBINE HEADS back to original format
# (B, H, S, dk) -> (B, S, H, dk) -> (B, S, D)
out = out.permute(0, 2, 1, 3)  # (B, S, H, dk)
print(f"After permute: {out.shape}")

# Reshape to combine heads: (B, S, H*dk) = (B, S, D)
out = out.contiguous().view(B, S, D)
print(f"Final attention output shape: {out.shape}")

After permute: torch.Size([2, 16, 8, 8])
Final attention output shape: torch.Size([2, 16, 64])


### 7) Top-k, sort, argmax operations

In [61]:
# Create example tensor for ranking operations
scores = torch.randn(3, 10)  # (Batch=3, Features=10)
print(f"Scores shape: {scores.shape}")
print(f"Sample scores:\n{scores[0]}")

Scores shape: torch.Size([3, 10])
Sample scores:
tensor([ 0.2670,  0.9228, -1.3007, -1.0200, -0.3297,  0.3023, -1.0869,  0.0442,
        -0.3207, -0.5611])


In [62]:
# TOP-K selection
k = 3
top_vals, top_indices = torch.topk(scores, k=k, dim=-1)  # Get top 3 per batch
print(f"Top-{k} values shape: {top_vals.shape}")  # (3, 3)
print(f"Top-{k} indices shape: {top_indices.shape}")  # (3, 3)
print(f"Top-{k} values for first batch: {top_vals[0]}")
print(f"Top-{k} indices for first batch: {top_indices[0]}")

Top-3 values shape: torch.Size([3, 3])
Top-3 indices shape: torch.Size([3, 3])
Top-3 values for first batch: tensor([0.9228, 0.3023, 0.2670])
Top-3 indices for first batch: tensor([1, 5, 0])


In [63]:
# SORTING (returns all elements sorted)
sorted_vals, sorted_indices = torch.sort(scores, dim=-1, descending=True)
print(f"Sorted values shape: {sorted_vals.shape}")  # Same as input
print(f"Sorted indices shape: {sorted_indices.shape}")  # Same as input

Sorted values shape: torch.Size([3, 10])
Sorted indices shape: torch.Size([3, 10])


### 8) Sparse tensors and irregular data

In [64]:
# SPARSE COO (COOrdinate format) tensor
# Useful for tensors with many zero elements
# Specified by: indices (coordinates) and values at those coordinates

# Create sparse tensor: [[3, 0, 0], [0, 4, 5]]
indices = torch.tensor([[0, 1, 1],  # Row indices
                        [0, 1, 2]])  # Column indices
values = torch.tensor([3.0, 4.0, 5.0])  # Values at those positions
sparse_shape = (2, 3)  # Dense tensor would be 2x3

In [65]:
sparse_tensor = torch.sparse_coo_tensor(indices, values, sparse_shape)
print(f"Sparse tensor indices shape: {indices.shape}")
print(f"Sparse tensor values shape: {values.shape}")
print(f"Sparse tensor logical shape: {sparse_tensor.shape}")
print(f"Sparse tensor:\n{sparse_tensor}")

# Convert to dense for verification
dense_version = sparse_tensor.to_dense()
print(f"Dense version:\n{dense_version}")

# SPARSE OPERATIONS
# Many operations preserve sparsity
sparse_scaled = sparse_tensor * 2
print(f"Sparse tensor * 2:\n{sparse_scaled.to_dense()}")

Sparse tensor indices shape: torch.Size([2, 3])
Sparse tensor values shape: torch.Size([3])
Sparse tensor logical shape: torch.Size([2, 3])
Sparse tensor:
tensor(indices=tensor([[0, 1, 1],
                       [0, 1, 2]]),
       values=tensor([3., 4., 5.]),
       size=(2, 3), nnz=3, layout=torch.sparse_coo)
Dense version:
tensor([[3., 0., 0.],
        [0., 4., 5.]])
Sparse tensor * 2:
tensor([[ 6.,  0.,  0.],
        [ 0.,  8., 10.]])


### 9) Einsum patterns reference

In [67]:
# Setup example tensors for different einsum patterns
A = torch.randn(3, 4)      # Matrix A: (3, 4)
B = torch.randn(4, 5)      # Matrix B: (4, 5)
C = torch.randn(2, 3, 4)   # Batch of matrices: (2, 3, 4)
D = torch.randn(2, 4, 5)   # Batch of matrices: (2, 4, 5)

In [68]:
# MATRIX MULTIPLICATION: 'ij,jk->ik'
# i=rows of A, j=cols of A/rows of B, k=cols of B
matmul_result = torch.einsum('ij,jk->ik', A, B)
print(f"Matrix multiply 'ij,jk->ik': {A.shape} @ {B.shape} = {matmul_result.shape}")

Matrix multiply 'ij,jk->ik': torch.Size([3, 4]) @ torch.Size([4, 5]) = torch.Size([3, 5])


In [69]:
# BATCHED MATRIX MULTIPLICATION: 'bij,bjk->bik'
# b=batch, i=rows, j=inner dim, k=cols
batch_matmul = torch.einsum('bij,bjk->bik', C, D)
print(f"Batched matmul 'bij,bjk->bik': {C.shape} @ {D.shape} = {batch_matmul.shape}")

Batched matmul 'bij,bjk->bik': torch.Size([2, 3, 4]) @ torch.Size([2, 4, 5]) = torch.Size([2, 3, 5])


In [70]:
# ATTENTION PATTERNS
Q = torch.randn(2, 8, 16, 64)  # (batch, heads, seq_len, key_dim)
K = torch.randn(2, 8, 20, 64)  # (batch, heads, seq_len, key_dim)
V = torch.randn(2, 8, 20, 64)  # (batch, heads, seq_len, value_dim)

In [71]:
# Attention scores: 'bhqd,bhkd->bhqk'
# b=batch, h=heads, q=query_pos, k=key_pos, d=key_dim
attn_scores = torch.einsum('bhqd,bhkd->bhqk', Q, K)
print(f"Attention scores 'bhqd,bhkd->bhqk': {Q.shape} x {K.shape} = {attn_scores.shape}")

# Apply attention to values: 'bhqk,bhkv->bhqv'
# b=batch, h=heads, q=query_pos, k=key_pos, v=value_dim
attn_output = torch.einsum('bhqk,bhkv->bhqv', F.softmax(attn_scores, dim=-1), V)
print(f"Attention output 'bhqk,bhkv->bhqv': {attn_scores.shape} x {V.shape} = {attn_output.shape}")

Attention scores 'bhqd,bhkd->bhqk': torch.Size([2, 8, 16, 64]) x torch.Size([2, 8, 20, 64]) = torch.Size([2, 8, 16, 20])
Attention output 'bhqk,bhkv->bhqv': torch.Size([2, 8, 16, 20]) x torch.Size([2, 8, 20, 64]) = torch.Size([2, 8, 16, 64])
